# Live Hyperoptax dashboard sweep

This notebook adapts the Branin benchmark from `bo_design_study.ipynb` into three recorded studies. Each completed optimizer batch is committed synchronously through `SQLiteRecorder`, so a dashboard pointed at the printed database path updates while the cell is still running.


In [ ]:
from __future__ import annotations

import os
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5

import jax
import jax.numpy as jnp
import numpy as np

from hyperoptax import BayesianSearch, LinearSpace, RandomSearch
from hyperoptax.dashboard import (
    Direction,
    SQLiteRecorder,
    StudyConfig,
    StudyQueryService,
)

requested_path = os.environ.get("HYPEROPTAX_DASHBOARD_DB")
base_path = Path(requested_path or "dashboard-demo.sqlite3").resolve()
if base_path.exists() and requested_path is None:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    base_path = base_path.with_name(f"dashboard-demo-{stamp}.sqlite3")
DB_PATH = base_path
SWEEP_ID = str(uuid5(NAMESPACE_URL, f"hyperoptax:{DB_PATH.name}:branin"))
CALLBACK_DELAY_SECONDS = 0.25

print(f"Recording dashboard studies in {DB_PATH}", flush=True)
print(f"Start the viewer with: hyperoptax dashboard {DB_PATH}", flush=True)

In [ ]:
def branin(x1: jax.Array, x2: jax.Array) -> jax.Array:
    """Branin on x1 in [-5, 10], x2 in [0, 15]."""
    b = 5.1 / (4.0 * jnp.pi**2)
    c = 5.0 / jnp.pi
    return (
        (x2 - b * x1**2 + c * x1 - 6.0) ** 2
        + 10.0 * (1.0 - 1.0 / (8.0 * jnp.pi)) * jnp.cos(x1)
        + 10.0
    )


def objective(_key: jax.Array, config: dict[str, jax.Array]) -> jax.Array:
    return -branin(config["x1"], config["x2"])


SPACE = {
    "x1": LinearSpace(-5.0, 10.0),
    "x2": LinearSpace(0.0, 15.0),
}
SPACE_SCHEMA = {
    "x1": {"type": "linear", "lower": -5.0, "upper": 10.0},
    "x2": {"type": "linear", "lower": 0.0, "upper": 15.0},
}

In [ ]:
def run_study(
    *,
    name: str,
    variant: str,
    seed: int,
    optimizer_name: str,
    optimizer_config: dict[str, int | float | bool],
    n_iterations: int,
) -> tuple[str, float]:
    n_parallel = int(optimizer_config["n_parallel"])
    if optimizer_name == "RandomSearch":
        state, optimizer = RandomSearch.init(SPACE, n_parallel=n_parallel)
    else:
        state, optimizer = BayesianSearch.init(
            SPACE,
            n_max=n_iterations * n_parallel,
            **optimizer_config,
        )

    study = StudyConfig(
        name=name,
        sweep_id=SWEEP_ID,
        direction=Direction.MAXIMIZE,
        optimizer_name=optimizer_name,
        optimizer_config=optimizer_config,
        search_space=SPACE_SCHEMA,
        n_parallel=n_parallel,
        seed=seed,
        variant=variant,
        replicate=str(seed),
        tags=("dashboard-demo", "branin"),
        metadata={"objective": "negative Branin", "true_best": -0.397887},
    )

    with SQLiteRecorder(DB_PATH, study=study) as recorder:

        def record_and_pause(event):
            recorder(event)
            completed = (event.batch_index + 1) * n_parallel
            print(
                f"\r{name}: {completed:>3}/{n_iterations * n_parallel} trials",
                end="",
                flush=True,
            )
            time.sleep(CALLBACK_DELAY_SECONDS)

        _, (_, results) = optimizer.optimize(
            state,
            jax.random.PRNGKey(seed),
            objective,
            n_iterations=n_iterations,
            callback=record_and_pause,
        )

    best = float(np.max(np.concatenate([np.asarray(batch) for batch in results])))
    print(f"  best={best:.6f}  study={recorder.study_id}", flush=True)
    return recorder.study_id, best

In [ ]:
RUNS = (
    {
        "name": "Branin · random batch 4",
        "variant": "random-n4",
        "seed": 7,
        "optimizer_name": "RandomSearch",
        "optimizer_config": {"n_parallel": 4},
        "n_iterations": 20,
    },
    {
        "name": "Branin · Bayesian serial",
        "variant": "bayesian-n1",
        "seed": 11,
        "optimizer_name": "BayesianSearch",
        "optimizer_config": {
            "n_parallel": 1,
            "n_warmup": 3,
            "n_candidates": 128,
            "n_restarts": 1,
            "n_lbfgs_steps": 5,
            "n_hparam_steps": 3,
        },
        "n_iterations": 24,
    },
    {
        "name": "Branin · Bayesian batch 4",
        "variant": "bayesian-n4",
        "seed": 23,
        "optimizer_name": "BayesianSearch",
        "optimizer_config": {
            "n_parallel": 4,
            "n_warmup": 3,
            "n_candidates": 128,
            "n_restarts": 1,
            "n_lbfgs_steps": 5,
            "n_hparam_steps": 3,
        },
        "n_iterations": 12,
    },
)

summaries = [run_study(**run) for run in RUNS]

In [ ]:
service = StudyQueryService(DB_PATH)
for study in service.list_studies():
    description = service.describe_study(study.study_id)
    trial_count = sum(description.trial_counts.values())
    best = description.best_trial.objective_value if description.best_trial else None
    best_text = "n/a" if best is None else f"{best:.6f}"
    print(
        f"{study.name:<32} {study.status.value:<10} "
        f"trials={trial_count:<3} best={best_text}"
    )
service.close()
print(f"\nDashboard database ready: {DB_PATH}")

## Open the result

Run `hyperoptax dashboard <printed-path>` in another terminal. Set `HYPEROPTAX_DASHBOARD_DB` before execution to append fresh studies to a particular database. On a remote host, keep the server bound to loopback and forward it with `ssh -N -L 8080:127.0.0.1:8080 user@host`.
